# Watermark Removal — ProPainter
JSON'dan watermark bölgelerini okur, tek mask oluşturur, ProPainter ile siler.

## 1 — Config + Kurulum

In [ ]:
import os, json, time, shutil, subprocess, gc
import cv2
import numpy as np
import torch
from pathlib import Path
from IPython.display import Video, display, HTML

# --- Drive ---
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

# --- Yollar ---
PIPELINE_ROOT = Path("/content/drive/MyDrive/pipeline")
JSON_PATH     = PIPELINE_ROOT / "watermark_detected" / "results.json"
OUTPUT_DIR    = PIPELINE_ROOT / "watermark_removed"
WORK_DIR      = Path("/content/wm_work")
PP_ROOT       = Path("/content/ProPainter")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# --- VRAM ayari ---
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# --- Parametreler ---
MASK_DILATION = 5       # manuel mask genisletme (piksel)
PP_NEIGHBOR   = 10
PP_REF_STRIDE = 10
PP_SUBVIDEO   = 80      # OOM onlemek icin 80'den dusuruldu
PP_RAFT_ITER  = 15      # statik watermark icin 20 overkill
PP_MAX_SIZE   = 960     # uzun kenar bu pikseli gecerse kucultulur
OOM_FALLBACK  = [720, 540]  # OOM olursa kademeli kucultme

# --- ProPainter Kurulum ---
if not PP_ROOT.exists():
    !git clone https://github.com/sczhou/ProPainter.git {PP_ROOT}

!pip install -q -r {PP_ROOT}/requirements.txt
!pip install -q timm einops av

WEIGHTS_DIR = PP_ROOT / "weights"
WEIGHTS_DIR.mkdir(exist_ok=True)
WEIGHT_URLS = {
    "ProPainter.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/ProPainter.pth",
    "recurrent_flow_completion.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/recurrent_flow_completion.pth",
    "raft-things.pth": "https://huggingface.co/camenduru/ProPainter/resolve/main/raft-things.pth",
    "i3d_rgb_imagenet.pt": "https://huggingface.co/camenduru/ProPainter/resolve/main/i3d_rgb_imagenet.pt",
}
for fname, url in WEIGHT_URLS.items():
    fpath = WEIGHTS_DIR / fname
    if not fpath.exists():
        print(f"Indiriliyor: {fname}")
        !wget -q -L -O {fpath} {url}

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")
print(f"Input : {JSON_PATH}")
print(f"Output: {OUTPUT_DIR}")

## 2 — JSON Oku

In [ ]:
with open(JSON_PATH) as f:
    data = json.load(f)

videos     = data["videos"]
wm_list    = {k: v for k, v in videos.items() if v["status"] == "WATERMARKED"}
clean_list = {k: v for k, v in videos.items() if v["status"] == "CLEAN"}

print(f"Toplam: {len(videos)} | WATERMARKED: {len(wm_list)} | CLEAN: {len(clean_list)}")

## 3 — Fonksiyonlar

In [ ]:
def create_mask(regions, width, height):
    """Tum watermark bolgelerini tek mask PNG'ye cizer."""
    mask = np.zeros((height, width), dtype=np.uint8)
    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        mask[y1:y2, x1:x2] = 255
    if MASK_DILATION > 0:
        kernel = np.ones((MASK_DILATION, MASK_DILATION), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
    return mask


def calc_resize_ratio(w, h, max_size=None):
    """Uzun kenar max_size'i geciyorsa kucultme orani hesapla."""
    if max_size is None:
        max_size = PP_MAX_SIZE
    long_side = max(w, h)
    if long_side <= max_size:
        return 1.0
    return round(max_size / long_side, 2)


def run_propainter(video_path, mask_path, output_dir, resize_ratio=1.0):
    """ProPainter inference: video + mask -> inpaint_out.mp4"""
    cmd = [
        "python", str(PP_ROOT / "inference_propainter.py"),
        "--video", str(video_path),
        "--mask", str(mask_path),
        "--output", str(output_dir),
        "--mask_dilation", "0",
        "--resize_ratio", str(resize_ratio),
        "--neighbor_length", str(PP_NEIGHBOR),
        "--ref_stride", str(PP_REF_STRIDE),
        "--subvideo_length", str(PP_SUBVIDEO),
        "--raft_iter", str(PP_RAFT_ITER),
        "--fp16",
    ]
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PP_ROOT), env=env)
    if result.returncode != 0:
        stderr = result.stderr[-1500:]
        if "OutOfMemoryError" in stderr or "out of memory" in stderr.lower():
            raise MemoryError(f"CUDA OOM (ratio={resize_ratio})")
        print(f"  STDOUT: {result.stdout[-1000:]}")
        print(f"  STDERR: {stderr}")
        raise RuntimeError("ProPainter basarisiz")


def mux_audio(inpaint_video, original_video, output_path):
    """Inpaint sonucuna orijinal sesi ekle."""
    result = subprocess.run([
        "ffmpeg", "-y",
        "-i", str(inpaint_video),
        "-i", str(original_video),
        "-map", "0:v", "-map", "1:a?",
        "-c:v", "copy", "-c:a", "copy",
        str(output_path),
        "-loglevel", "error"
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg mux basarisiz: {result.stderr[-300:]}")


def remove_watermark(video_path, regions, output_path):
    """Tek video icin: mask olustur -> ProPainter -> ses ekle.
       OOM olursa kademeli olarak kucultup tekrar dener."""
    t0 = time.time()
    pp_out = WORK_DIR / "pp_output"
    mask_path = WORK_DIR / "mask.png"

    try:
        # Video boyutlarini oku
        cap = cv2.VideoCapture(str(video_path))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        cap.release()

        # Tek mask olustur (tum bolgeler birlesik)
        mask = create_mask(regions, w, h)
        cv2.imwrite(str(mask_path), mask)

        # Denenecek max_size degerleri: PP_MAX_SIZE, sonra fallback'ler
        sizes_to_try = [PP_MAX_SIZE] + OOM_FALLBACK
        last_err = None

        for attempt, max_sz in enumerate(sizes_to_try):
            ratio = calc_resize_ratio(w, h, max_sz)
            eff_w, eff_h = int(w * ratio), int(h * ratio)

            if attempt == 0:
                print(f"  {w}x{h}, {len(regions)} bolge, resize={ratio} ({eff_w}x{eff_h})")
            else:
                print(f"  OOM retry {attempt}: max_size={max_sz}, resize={ratio} ({eff_w}x{eff_h})")

            # Temizlik
            if pp_out.exists():
                shutil.rmtree(pp_out)
            gc.collect()
            torch.cuda.empty_cache()

            try:
                run_propainter(video_path, mask_path, pp_out, ratio)
                last_err = None
                break  # basarili
            except MemoryError as e:
                last_err = e
                print(f"  OOM: {e}")
                continue

        if last_err is not None:
            raise last_err

        # Sonuc: pp_output/inpaint_out.mp4
        inpaint_mp4 = pp_out / "inpaint_out.mp4"
        if not inpaint_mp4.exists():
            # Fallback: klasorde mp4 ara
            mp4s = list(pp_out.rglob("*.mp4"))
            mp4s = [m for m in mp4s if "mask" not in m.name.lower()]
            if not mp4s:
                raise FileNotFoundError(f"ProPainter ciktisi bulunamadi: {pp_out}")
            inpaint_mp4 = mp4s[0]

        # Ses muxla
        mux_audio(inpaint_mp4, video_path, output_path)

        elapsed = time.time() - t0
        size_mb = Path(output_path).stat().st_size / (1024 * 1024)
        print(f"  Tamamlandi: {elapsed:.1f}sn, {size_mb:.1f}MB")
        return elapsed

    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        return -1

    finally:
        if pp_out.exists(): shutil.rmtree(pp_out)
        if mask_path.exists(): mask_path.unlink()
        gc.collect()
        torch.cuda.empty_cache()


print("Fonksiyonlar hazir.")

## 4 — Calistir

In [ ]:
t_start = time.time()
results = []
total = len(wm_list) + len(clean_list)

for i, (name, info) in enumerate(wm_list.items(), 1):
    print(f"\n[{i}/{total}] PP: {name}")
    elapsed = remove_watermark(info["source_path"], info["regions"], str(OUTPUT_DIR / name))
    results.append({"video": name, "action": "PP" if elapsed > 0 else "HATA",
                    "regions": len(info["regions"]), "seconds": round(max(elapsed, 0), 1)})

for i, (name, info) in enumerate(clean_list.items(), len(wm_list) + 1):
    print(f"[{i}/{total}] COPY: {name}")
    shutil.copy2(info["source_path"], str(OUTPUT_DIR / name))
    results.append({"video": name, "action": "COPY", "regions": 0, "seconds": 0})

# --- Ozet ---
print(f"\n{'='*60}")
print(f"{'Video':<45} {'Islem':<6} {'Bolge':>5} {'Sure':>8}")
print("-" * 65)
for r in results:
    sure = f"{r['seconds']}sn" if r['seconds'] > 0 else "-"
    print(f"{r['video']:<45} {r['action']:<6} {r['regions']:>5} {sure:>8}")

n_pp  = sum(1 for r in results if r['action'] == 'PP')
n_err = sum(1 for r in results if r['action'] == 'HATA')
print(f"\n{n_pp} inpaint | {len(clean_list)} kopya | {n_err} hata | Toplam: {time.time()-t_start:.1f}sn")

## 5 — Preview

In [ ]:
gc.collect()
torch.cuda.empty_cache()

output_videos = sorted(f for f in OUTPUT_DIR.iterdir() if f.suffix.lower() in ('.mp4','.mov','.avi','.mkv'))
print(f"{len(output_videos)} video hazir:")
for i, f in enumerate(output_videos):
    tag = "PP" if f.name in wm_list else "CLEAN"
    print(f"  {i}: {f.name} [{tag}]")

def preview(n):
    f = output_videos[n]
    local = str(WORK_DIR / f.name)
    shutil.copy2(str(f), local)
    display(HTML(f"<h3>{f.name}</h3>"))
    display(Video(local, embed=True, width=720))
    os.remove(local)

# preview(0)